# Imaging Results from Dynamics Explorer 1 — Implementation / 구현

**Paper**: Frank, L. A., and J. D. Craven, *Imaging Results from Dynamics Explorer 1*, Reviews of Geophysics, 26(2), 249–283, 1988. DOI: 10.1029/RG026i002p00249

This notebook implements three core concepts that appear in the paper:
1. **Spin-scan imaging cadence model** — how the spacecraft spin (10 rpm) and mirror step build a 2D image in 12 minutes (§1, Figure 1).
2. **Auroral oval geometry projection** — projecting an idealized oval at 65–75° magnetic latitude into the spacecraft's apparent disk view from 3.65 R_E apogee, with terminator overlay (§1–§2, e.g. Figs 3, 4, 19).
3. **Photon counting / dayglow background model with Poisson statistics** — the statistical framework Frank et al. (1986b/c) used to identify atmospheric "holes" as departures from Poisson dayglow (§7, Figure 40).

이 노트북은 논문의 세 핵심 개념을 구현한다:
1. **스핀 스캔 영상 주기 모형** — 위성 스핀(10 rpm)과 거울 스텝이 어떻게 12분만에 2D 영상을 만드는가(§1, Figure 1).
2. **오로라 오벌 기하 투영** — 자기위도 65–75°의 이상화된 오벌을 원지점 3.65 R_E의 위성 시야 디스크에 투영, 터미네이터 오버레이 포함(§1–§2).
3. **광자 계수 및 데이글로우 배경 모형 + Poisson 통계** — Frank et al.(1986b/c)이 대기 구멍을 Poisson 데이글로우로부터의 이탈로 식별한 통계 프레임(§7, Figure 40).

All code uses Google-style English docstrings. Plots and prose remain bilingual.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.collections import LineCollection
from scipy.stats import poisson, norm

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Earth and orbital constants used throughout
R_EARTH_KM = 6371.0
DE1_APOGEE_RE = 3.65          # Earth radii at apogee
DE1_APOGEE_KM = DE1_APOGEE_RE * R_EARTH_KM
DE1_PERIGEE_KM = 570.0
DE1_INCLINATION_DEG = 90.0    # polar orbit
DE1_PERIOD_HR = 6.83          # orbital period
DE1_SPIN_RPM = 10.0
DE1_SPIN_PERIOD_S = 60.0 / DE1_SPIN_RPM  # 6 s per rotation

## Part 1: Spin-Scan Imaging Cadence Model / 스핀 스캔 영상 주기 모형

**EN**: Each spacecraft spin (period $T_{\rm spin}=6$ s) sweeps the photometer's instantaneous field of view (IFOV) along one scan line. An internal rotating mirror then steps the IFOV perpendicular to the scan direction by an angular increment $\Delta\theta_{\rm step}$ (typically 0.25°). After $N_{\rm rot}$ rotations the image covers $N_{\rm rot}\cdot\Delta\theta_{\rm step}$ in the cross-scan direction. The total acquisition time is

$$T_{\rm frame} = N_{\rm rot}\cdot T_{\rm spin}.$$

For $N_{\rm rot}=120$, $T_{\rm spin}=6$ s, $\Delta\theta_{\rm step}=0.25°$ ⇒ $T_{\rm frame}=720$ s = 12 min covering 30°.

**KR**: 한 번의 위성 스핀(주기 6초)이 광도계의 즉시 시야(IFOV)를 한 스캔라인을 따라 쓸어간다. 내부 회전 거울이 스핀마다 시야를 직각 방향으로 0.25°씩 옮긴다. $N_{\rm rot}$ 회전 뒤 영상은 $N_{\rm rot}\cdot\Delta\theta_{\rm step}$ 폭을 덮으며 총 시간은 위 식과 같다. $N_{\rm rot}=120$, 6초, 0.25°일 때 720초 = 12분, 30° 폭.

In [ ]:
def spin_scan_cadence(
    spin_rpm: float = 10.0,
    n_rotations: int = 120,
    pixel_angular_size_deg: float = 0.25,
    mirror_step_deg: float = 0.25,
) -> dict:
    """Compute spin-scan imaging timing parameters.

    Args:
        spin_rpm: Spacecraft spin rate in revolutions per minute.
        n_rotations: Number of spacecraft rotations per image frame.
        pixel_angular_size_deg: Angular pixel size in degrees.
        mirror_step_deg: Mirror step between rotations in degrees.

    Returns:
        Dictionary with keys: spin_period_s, frame_time_s, image_width_deg,
        scan_line_pixels (per spin), pixel_dwell_ms.
    """
    spin_period = 60.0 / spin_rpm
    frame_time = n_rotations * spin_period
    image_width = n_rotations * mirror_step_deg
    # Spin sweeps 360 deg in one period; useful angular range across Earth's
    # disk depends on the geometry, but per-pixel dwell is set by spin rate.
    omega_spin_deg_per_s = 360.0 / spin_period
    pixel_dwell_ms = 1000.0 * pixel_angular_size_deg / omega_spin_deg_per_s
    # The scan line across a 30 deg target spans 30/pixel_size pixels.
    scan_line_pixels = int(round(image_width / pixel_angular_size_deg))
    return {
        "spin_period_s": spin_period,
        "frame_time_s": frame_time,
        "frame_time_min": frame_time / 60.0,
        "image_width_deg": image_width,
        "scan_line_pixels": scan_line_pixels,
        "pixel_dwell_ms": pixel_dwell_ms,
    }


# Reproduce DE 1 nominal parameters from Frank et al. (1981a) / Figure 1
nominal = spin_scan_cadence()
print("DE 1 nominal spin-scan parameters / 공칭 매개변수:")
for k, v in nominal.items():
    if isinstance(v, float):
        print(f"  {k:22s} = {v:.3f}")
    else:
        print(f"  {k:22s} = {v}")

# Two-photometer combined mode reaches ~2 min per image at reduced FOV
fast = spin_scan_cadence(n_rotations=20)
print("\nReduced-FOV fast mode (n_rotations=20) / 축소 FOV 고속 모드:")
print(f"  frame time = {fast['frame_time_s']:.0f} s = {fast['frame_time_min']:.1f} min")

In [ ]:
def visualize_spin_scan_buildup(
    n_rotations: int = 120,
    pixel_size_deg: float = 0.25,
    show_every: int = 20,
) -> None:
    """Visualize how the 2D image is built up over successive rotations.

    Args:
        n_rotations: Total rotations per image frame.
        pixel_size_deg: Single pixel angular size in degrees.
        show_every: Plot snapshots at this rotation interval.
    """
    fig, axes = plt.subplots(2, 3, figsize=(13, 8.5))
    snapshots = list(range(show_every, n_rotations + 1, show_every))[:6]
    image_width_pix = int(n_rotations * pixel_size_deg / pixel_size_deg)

    # Build a synthetic auroral target: ring with 65-75 deg latitude analogue
    grid = np.zeros((image_width_pix, image_width_pix))
    cx, cy = image_width_pix / 2, image_width_pix / 2
    yy, xx = np.indices(grid.shape)
    rr = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
    target = np.exp(-0.5 * ((rr - 35) / 4) ** 2) * 5.0
    target += np.random.RandomState(42).normal(0.5, 0.1, target.shape)

    for ax, n_done in zip(axes.flat, snapshots):
        partial = np.zeros_like(grid)
        partial[:n_done] = target[:n_done]
        im = ax.imshow(partial, cmap='inferno', origin='lower', vmin=0, vmax=6)
        elapsed_s = n_done * 6.0
        ax.set_title(
            f"After {n_done} rotations / {n_done}회전 후\n"
            f"t = {elapsed_s:.0f} s ({elapsed_s/60:.1f} min)"
        )
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(
        "Spin-scan image buildup over 120 rotations\n"
        "스핀 스캔 영상이 120 회전에 걸쳐 형성됨",
        fontsize=13,
    )
    plt.tight_layout()
    plt.show()


visualize_spin_scan_buildup()

**Interpretation / 해석**: Each panel shows how additional scan lines fill in the image as the spacecraft completes more rotations. With $T_{\rm spin}=6$ s, 120 rotations corresponds to 12 minutes of acquisition — long compared to fast substorm dynamics (which can change on 1-min timescales) but short compared to the 6.83-h orbital period.

각 패널은 위성이 더 많이 회전할수록 추가 스캔라인이 영상을 채워가는 모습을 보여준다. 6초 × 120회전 = 12분은 빠른 서브스톰 동역학(분 단위)에 비해 길지만, 6.83시간 궤도 주기에 비해 짧다.

## Part 2: Auroral Oval Geometry Projection / 오로라 오벌 기하 투영

**EN**: From DE 1's high-altitude apogee at 3.65 R_E, the entire visible Earth disk subtends an angular diameter of
$$\theta_{\rm disk} = 2\arcsin(R_\oplus / r_{\rm sat}),$$
where $r_{\rm sat} = 3.65\, R_\oplus$. This gives $\theta_{\rm disk}\approx 31.6°$, comfortably within the 30°×30° field of view. We project an idealized auroral oval (centered on the magnetic pole at $\sim 79°$N geographic, with mean radius 65–75° magnetic latitude) onto the apparent disk, with a terminator (solar zenith angle = 98°, the night-edge for VUV airglow) overlaid. This reproduces the geometric concept of Frank-Craven Figures 3, 4, 7, 19.

**KR**: 원지점 3.65 R_E 고도에서 보이는 지구 디스크의 각직경은 위 식으로 약 31.6°이며, 30°×30° 시야 안에 안락히 들어간다. 자기극(약 79°N) 중심에 평균 위도 65–75°의 이상화된 오로라 오벌을 투영하고, 태양 천정각 98°(VUV 대기광 야간 가장자리) 터미네이터를 겹쳐 그린다. 이는 Figure 3, 4, 7, 19의 기하 개념을 재현한다.

In [ ]:
def earth_angular_diameter_deg(spacecraft_radius_re: float) -> float:
    """Compute Earth's angular diameter as seen from a spacecraft.

    Args:
        spacecraft_radius_re: Spacecraft geocentric distance in Earth radii.

    Returns:
        Angular diameter of Earth's disk in degrees.
    """
    return 2.0 * np.degrees(np.arcsin(1.0 / spacecraft_radius_re))


for r_re in [1.5, 2.0, 3.0, 3.65, 5.0, 8.0]:
    print(
        f"r = {r_re:.2f} R_E -> Earth disk diameter = "
        f"{earth_angular_diameter_deg(r_re):5.2f} deg"
    )

In [ ]:
def latlon_to_eci(lat_deg: np.ndarray, lon_deg: np.ndarray, alt_km: float = 0.0) -> np.ndarray:
    """Convert geographic (or geomagnetic) lat/lon to ECI Cartesian coordinates.

    Args:
        lat_deg: Latitude in degrees, shape (N,).
        lon_deg: Longitude in degrees, shape (N,).
        alt_km: Altitude above sphere in km (default 0 = surface).

    Returns:
        Array of shape (N, 3) with x, y, z in km.
    """
    lat = np.radians(lat_deg)
    lon = np.radians(lon_deg)
    r = R_EARTH_KM + alt_km
    x = r * np.cos(lat) * np.cos(lon)
    y = r * np.cos(lat) * np.sin(lon)
    z = r * np.sin(lat)
    return np.stack([x, y, z], axis=-1)


def project_to_camera_plane(
    points_eci: np.ndarray, sat_eci: np.ndarray
) -> tuple:
    """Project ECI points onto the spacecraft camera plane.

    The camera looks from sat_eci toward Earth's center. Coordinates are in
    a tangent plane at Earth center, with x along the projected satellite
    east-direction and y along projected north.

    Args:
        points_eci: ECI Cartesian points, shape (N, 3) in km.
        sat_eci: Spacecraft ECI position, shape (3,) in km.

    Returns:
        Tuple (x_img, y_img, visible_mask). x_img, y_img are angular
        coordinates in degrees on the camera plane; visible_mask is a
        boolean array indicating which points are on the near hemisphere.
    """
    los = -sat_eci / np.linalg.norm(sat_eci)  # Earth-pointing unit vector
    # Build orthonormal camera frame with z = Earth-pointing
    z_hat = los
    # Choose x_hat in the plane perpendicular to spin axis (assume Earth's z)
    earth_z = np.array([0.0, 0.0, 1.0])
    x_hat = np.cross(earth_z, z_hat)
    if np.linalg.norm(x_hat) < 1e-9:
        x_hat = np.array([1.0, 0.0, 0.0])
    x_hat /= np.linalg.norm(x_hat)
    y_hat = np.cross(z_hat, x_hat)

    rel = points_eci - sat_eci  # vector from satellite to point
    dist = np.linalg.norm(rel, axis=-1)
    rel_unit = rel / dist[:, None]
    # Angular offsets in radians
    ax = np.arcsin(np.dot(rel_unit, x_hat))
    ay = np.arcsin(np.dot(rel_unit, y_hat))
    # Visibility: dot product with Earth-center-from-point > 0 (near hemisphere)
    point_outward = points_eci / np.linalg.norm(points_eci, axis=-1)[:, None]
    visible = np.einsum('ij,ij->i', point_outward, -rel_unit) > 0
    return np.degrees(ax), np.degrees(ay), visible


def auroral_oval_points(
    pole_lat_deg: float = 79.0,
    pole_lon_deg: float = -71.0,
    eq_lat_deg: float = 65.0,
    pol_lat_deg: float = 75.0,
    n_points: int = 360,
) -> tuple:
    """Generate equatorward and poleward boundary curves of an idealized oval.

    The oval is treated as a circle in geomagnetic coordinates (offset from
    geographic pole) at fixed magnetic colatitude.

    Args:
        pole_lat_deg: Geographic latitude of the magnetic pole.
        pole_lon_deg: Geographic longitude of the magnetic pole.
        eq_lat_deg: Equatorward boundary geomagnetic latitude.
        pol_lat_deg: Poleward boundary geomagnetic latitude.
        n_points: Number of points along each boundary.

    Returns:
        Tuple (eq_geo_lat, eq_geo_lon, pol_geo_lat, pol_geo_lon) in degrees.
    """
    mlt_rad = np.linspace(0.0, 2.0 * np.pi, n_points)

    def mag_to_geo(mlat_deg: float, mlt_rad: np.ndarray) -> tuple:
        # Spherical rotation: magnetic pole at (pole_lat_deg, pole_lon_deg)
        mcolat = np.radians(90.0 - mlat_deg)
        # In the magnetic frame, point is at colatitude mcolat, azimuth = mlt
        x_m = np.sin(mcolat) * np.cos(mlt_rad)
        y_m = np.sin(mcolat) * np.sin(mlt_rad)
        z_m = np.cos(mcolat) * np.ones_like(mlt_rad)
        # Rotate so magnetic z aligns with geographic pole direction
        plat = np.radians(pole_lat_deg)
        plon = np.radians(pole_lon_deg)
        # Rotation: first rotate around y by (90 - plat), then around z by plon
        cy, sy = np.cos(np.pi / 2 - plat), np.sin(np.pi / 2 - plat)
        cz, sz = np.cos(plon), np.sin(plon)
        # Rotation around y
        x1 = cy * x_m + sy * z_m
        y1 = y_m
        z1 = -sy * x_m + cy * z_m
        # Rotation around z
        x2 = cz * x1 - sz * y1
        y2 = sz * x1 + cz * y1
        z2 = z1
        # Convert back to lat/lon
        glat = np.degrees(np.arcsin(np.clip(z2, -1.0, 1.0)))
        glon = np.degrees(np.arctan2(y2, x2))
        return glat, glon

    eq_lat, eq_lon = mag_to_geo(eq_lat_deg, mlt_rad)
    pol_lat, pol_lon = mag_to_geo(pol_lat_deg, mlt_rad)
    return eq_lat, eq_lon, pol_lat, pol_lon


def plot_oval_view_from_de1(
    sat_lat_deg: float = 78.2,
    sat_lon_deg: float = 0.0,
    sat_alt_re: float = DE1_APOGEE_RE,
    sun_lon_deg: float = 0.0,
) -> None:
    """Plot an idealized auroral oval as seen from a DE 1-like apogee position.

    Args:
        sat_lat_deg: Spacecraft sub-point latitude in degrees.
        sat_lon_deg: Spacecraft sub-point longitude in degrees.
        sat_alt_re: Spacecraft geocentric distance in Earth radii.
        sun_lon_deg: Sub-solar longitude in degrees (defines terminator).
    """
    # Spacecraft position in ECI
    sat_eci = latlon_to_eci(
        np.array([sat_lat_deg]),
        np.array([sat_lon_deg]),
        alt_km=(sat_alt_re - 1) * R_EARTH_KM,
    )[0]

    # Earth limb (great-circle approximation: points at the visible-edge angle)
    half_angle = np.arcsin(1.0 / sat_alt_re)
    sat_unit = sat_eci / np.linalg.norm(sat_eci)
    perp1 = np.cross(sat_unit, np.array([0.0, 0.0, 1.0]))
    perp1 /= np.linalg.norm(perp1)
    perp2 = np.cross(sat_unit, perp1)
    azimuths = np.linspace(0.0, 2.0 * np.pi, 200)
    limb_pts = (
        np.cos(half_angle) * sat_unit
        + np.sin(half_angle) * (
            np.cos(azimuths)[:, None] * perp1
            + np.sin(azimuths)[:, None] * perp2
        )
    ) * R_EARTH_KM

    # Auroral oval boundaries
    eq_lat, eq_lon, pol_lat, pol_lon = auroral_oval_points()
    eq_eci = latlon_to_eci(eq_lat, eq_lon)
    pol_eci = latlon_to_eci(pol_lat, pol_lon)

    # Project everything
    limb_x, limb_y, _ = project_to_camera_plane(limb_pts, sat_eci)
    eq_x, eq_y, eq_vis = project_to_camera_plane(eq_eci, sat_eci)
    pol_x, pol_y, pol_vis = project_to_camera_plane(pol_eci, sat_eci)

    # Terminator: points at sub-solar plane perpendicular (lat/lon where SZA = 90)
    # For simplicity, treat sun in equatorial plane; terminator is great circle
    term_lon = np.linspace(-180.0, 180.0, 360)
    term_lat = np.degrees(np.arctan(
        -np.cos(np.radians(term_lon - sun_lon_deg))
        / np.tan(np.radians(0.0))  # Sun at equator
    )) if False else np.zeros_like(term_lon)  # use simple equatorial proxy
    # Simpler: terminator is a great circle 90 deg from sub-solar point
    sub_solar_eci = latlon_to_eci(np.array([0.0]), np.array([sun_lon_deg]))[0]
    sub_solar_unit = sub_solar_eci / np.linalg.norm(sub_solar_eci)
    # Two perpendicular vectors
    if abs(sub_solar_unit[2]) < 0.99:
        e1 = np.cross(sub_solar_unit, np.array([0.0, 0.0, 1.0]))
    else:
        e1 = np.cross(sub_solar_unit, np.array([1.0, 0.0, 0.0]))
    e1 /= np.linalg.norm(e1)
    e2 = np.cross(sub_solar_unit, e1)
    th = np.linspace(0.0, 2.0 * np.pi, 200)
    term_eci = R_EARTH_KM * (
        np.cos(th)[:, None] * e1 + np.sin(th)[:, None] * e2
    )
    term_x, term_y, term_vis = project_to_camera_plane(term_eci, sat_eci)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot(limb_x, limb_y, 'k-', lw=1.5, label="Earth limb / 지구 림")
    # Plot oval boundaries with visibility mask
    ax.plot(eq_x[eq_vis], eq_y[eq_vis], 'r-', lw=2,
            label="Equatorward boundary (65° MLAT) / 적도측 경계")
    ax.plot(pol_x[pol_vis], pol_y[pol_vis], 'b-', lw=2,
            label="Poleward boundary (75° MLAT) / 극측 경계")
    ax.plot(term_x[term_vis], term_y[term_vis], 'y--', lw=1.5,
            label="Terminator / 터미네이터")
    ax.set_xlabel("x angular (deg) / x 각도")
    ax.set_ylabel("y angular (deg) / y 각도")
    ax.set_title(
        f"DE 1 view from r = {sat_alt_re:.2f} R_E, sub-point ({sat_lat_deg}, {sat_lon_deg})\n"
        f"DE 1의 시야 — 원지점 3.65 R_E"
    )
    ax.set_aspect('equal')
    ax.set_xlim(-20, 20); ax.set_ylim(-20, 20)
    ax.legend(loc='lower right', fontsize=9)
    plt.tight_layout()
    plt.show()


plot_oval_view_from_de1()

**Interpretation / 해석**: The poleward (blue) and equatorward (red) boundaries form the auroral oval ring; the dashed yellow line is the day-night terminator. From DE 1 apogee, only one ring is visible at a time, with the dark sector visible to the spacecraft when the spacecraft is over the night side. This geometry is exactly what produces the crescent-shaped image of the oval seen in Figures 3, 4, and 19 of the paper.

극측(파란색)과 적도측(빨간색) 경계가 오로라 오벌 띠를 형성하고, 노란 점선은 주야 터미네이터다. DE 1 원지점에서는 한 번에 한 고리만 보이며, 위성이 야간 반구 위에 있을 때 어두운 구역이 위성 시야에 들어온다. 이 기하가 논문 Figure 3, 4, 19의 초승달 모양 오벌 영상을 만드는 원리다.

## Part 3: Photon Counting / Dayglow Background Model with Poisson Statistics / 광자 계수 및 데이글로우 배경 모형 + Poisson 통계

**EN**: The atmospheric "holes" claim of Frank et al. (1986b/c) hinged on a statistical argument: pixel counts in a 130.4 nm dayglow image should follow a Poisson distribution if the dayglow is uniform and the photometer is shot-noise-limited. Frank et al. (Figure 40) showed that the empirical distribution at altitudes 3.0–3.3 R_E has excess at $\leq -3\sigma$ (the dark holes) and $\geq +3\sigma$ (limb-brightening of the cometary water cloud). We reproduce this analysis with synthetic data:
1. Generate a baseline Poisson dayglow background.
2. Inject sparse "holes" — pixels with $\geq 80\%$ reduced count.
3. Compute the empirical PDF and compare to the pure-Poisson prediction.
4. Show the resulting excess at $\sigma \lesssim -3$, mirroring Figure 40.

**Important caveat / 중요 주의**: Modern Polar VIS analyses (Parks et al., 1998) show that the apparent excess is consistent with detector dark current, cosmic-ray hits, and 1-σ noise excursions of a near-detector-limit photon flux — not water vapor clouds. The simulation below is therefore a statistical demonstration of the methodology, not an endorsement of the small-comet hypothesis.

**KR**: Frank et al.(1986b/c)의 대기 구멍 주장은 통계적 논거에 달려 있다 — 균일한 데이글로우와 샷 잡음 한계의 광도계라면 130.4 nm 영상의 픽셀 계수는 Poisson 분포를 따라야 한다. Figure 40에서 3.0–3.3 R_E 고도의 경험 분포는 $\leq -3\sigma$(어두운 구멍)와 $\geq +3\sigma$(혜성 수증기 구름의 림 브라이트닝)에서 초과를 보였다. 우리는 합성 데이터로 다음을 재현한다:
1. 기본 Poisson 데이글로우 배경 생성.
2. ≥80% 감소된 "구멍" 픽셀 희소 삽입.
3. 경험 PDF를 순수 Poisson과 비교.
4. $\sigma \lesssim -3$의 초과를 시각화 — Figure 40 재현.

**중요 주의**: 현대 Polar VIS 분석(Parks et al., 1998)은 외관상 초과가 검출기 암전류, 우주선 충돌, 검출 한계 부근 광자 플럭스의 1-σ 잡음 변동과 일치함을 보였다 — 수증기 구름이 아니다. 아래 시뮬레이션은 방법론의 통계적 시연이지 소혜성 가설 지지가 아니다.

In [ ]:
def generate_dayglow_image_with_holes(
    image_size: int = 200,
    mean_counts: float = 44.0,
    hole_fraction: float = 0.001,
    hole_depth_factor: float = 0.2,
    rng_seed: int = 42,
) -> tuple:
    """Generate a synthetic 130.4 nm dayglow image with injected dark holes.

    Background follows Poisson(mean_counts). Holes are pixels whose mean is
    multiplied by hole_depth_factor (e.g. 0.2 = 80% reduced).

    Args:
        image_size: Square image side in pixels.
        mean_counts: Poisson rate parameter for ordinary dayglow pixels.
        hole_fraction: Fraction of pixels treated as holes.
        hole_depth_factor: Multiplier of the local mean inside holes.
        rng_seed: NumPy random seed for reproducibility.

    Returns:
        Tuple (image, hole_mask) where image is integer counts and
        hole_mask is a boolean array marking the injected holes.
    """
    rng = np.random.default_rng(rng_seed)
    means = np.full((image_size, image_size), mean_counts)
    n_pixels = image_size * image_size
    n_holes = int(round(hole_fraction * n_pixels))
    flat_idx = rng.choice(n_pixels, size=n_holes, replace=False)
    hole_mask = np.zeros(n_pixels, dtype=bool)
    hole_mask[flat_idx] = True
    hole_mask = hole_mask.reshape(image_size, image_size)
    means[hole_mask] *= hole_depth_factor
    image = rng.poisson(means)
    return image, hole_mask


def standardized_pixel_anomaly(image: np.ndarray, kernel_size: int = 7) -> np.ndarray:
    """Compute the standard-deviation anomaly of each pixel relative to its neighborhood.

    Each pixel's value is converted to (count - local_mean) / sqrt(local_mean),
    where local_mean is computed from a square window of size kernel_size
    centered on the pixel (excluding the pixel itself for fairness).

    Args:
        image: 2D integer count image.
        kernel_size: Side of the square neighborhood (odd integer).

    Returns:
        2D float array of standardized anomalies in units of sigma.
    """
    from scipy.ndimage import uniform_filter
    local_mean = uniform_filter(image.astype(float), size=kernel_size, mode='reflect')
    # Subtract self-contribution for an unbiased estimate (approx)
    n_neighbors = kernel_size * kernel_size
    unbiased = (n_neighbors * local_mean - image) / (n_neighbors - 1)
    return (image - unbiased) / np.sqrt(np.clip(unbiased, 1.0, None))


image, hole_mask = generate_dayglow_image_with_holes()
anomaly = standardized_pixel_anomaly(image)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image, cmap='inferno', origin='lower')
axes[0].set_title("Synthetic dayglow image / 합성 데이글로우 영상\n(130.4 nm Poisson + holes)")
axes[1].imshow(anomaly, cmap='seismic', origin='lower', vmin=-5, vmax=5)
axes[1].set_title("Standardized anomaly (sigma) / 표준화 이상\nDark spots = injected holes")
axes[2].imshow(hole_mask, cmap='gray', origin='lower')
axes[2].set_title("Ground-truth hole locations / 정답 구멍 위치")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

In [ ]:
def plot_anomaly_distribution(anomaly: np.ndarray, mean_counts: float = 44.0) -> None:
    """Compare the observed anomaly distribution against a pure-Poisson prediction.

    Args:
        anomaly: 2D array of standardized anomalies (sigma).
        mean_counts: Mean Poisson rate used for the theoretical comparison.
    """
    flat = anomaly.flatten()
    bins = np.linspace(-6, 6, 121)
    centers = 0.5 * (bins[:-1] + bins[1:])
    hist, _ = np.histogram(flat, bins=bins, density=True)
    # Pure Gaussian/Poisson expectation: standard normal in the large-mean limit
    gaussian = norm.pdf(centers)
    # Frank et al. style: occurrence frequency on log scale
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.plot(centers, hist, 'ko-', ms=4, label='Observed (with holes) / 관측 (구멍 포함)')
    ax.plot(centers, gaussian, 'b--', lw=2,
            label='Pure Poisson/Gaussian / 순수 Poisson')
    ax.set_yscale('log')
    ax.set_xlabel('Standardized anomaly σ / 표준화 이상 σ')
    ax.set_ylabel('Occurrence frequency / 발생 빈도')
    ax.set_title(
        "Pixel-count anomaly distribution (Frank et al. 1987b style)\n"
        "픽셀 계수 이상 분포"
    )
    ax.axvline(-3, color='red', ls=':', lw=1, label='-3σ threshold')
    ax.axvline(+3, color='orange', ls=':', lw=1, label='+3σ threshold')
    ax.set_ylim(1e-6, 1.0)
    ax.legend()
    plt.tight_layout()
    plt.show()


plot_anomaly_distribution(anomaly)

In [ ]:
def detection_efficiency_vs_hole_depth(
    mean_counts: float = 44.0,
    sigma_threshold: float = 3.0,
    image_size: int = 200,
) -> None:
    """Plot detection efficiency for atmospheric holes vs. hole depth.

    Reproduces the dependence of detection on the contrast between the
    occulting cloud and the dayglow background.

    Args:
        mean_counts: Mean Poisson dayglow counts per pixel.
        sigma_threshold: Detection threshold in sigma units.
        image_size: Image side in pixels for the simulation.
    """
    depths = np.linspace(0.05, 0.95, 19)
    efficiencies = []
    false_alarms = []
    for depth in depths:
        img, mask = generate_dayglow_image_with_holes(
            image_size=image_size,
            mean_counts=mean_counts,
            hole_fraction=0.005,
            hole_depth_factor=depth,
        )
        anom = standardized_pixel_anomaly(img)
        detected = anom < -sigma_threshold
        true_positive = (detected & mask).sum()
        false_positive = (detected & ~mask).sum()
        efficiencies.append(true_positive / max(mask.sum(), 1))
        false_alarms.append(false_positive / max((~mask).sum(), 1))

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(depths, efficiencies, 'go-', label='True-positive rate / 참 검출률')
    ax.plot(depths, false_alarms, 'ro-', label='False-alarm rate / 오탐률')
    ax.axhline(0, color='k', lw=0.5)
    ax.set_xlabel('Hole transparency factor (1.0 = no hole) / 구멍 투과 인자')
    ax.set_ylabel('Rate')
    ax.set_title(
        f"Hole detectability vs. contrast at -{sigma_threshold:.0f}σ threshold\n"
        f"-{sigma_threshold:.0f}σ 임계에서 구멍 검출 가능성 vs. 대비"
    )
    ax.legend()
    plt.tight_layout()
    plt.show()


detection_efficiency_vs_hole_depth()

**Interpretation / 해석**: The detection efficiency curves illustrate the fundamental difficulty Frank et al. faced: at hole depths ≳50% (factor ≤ 0.5) the true-positive rate climbs steeply, but for shallow contrast (≳70%, factor ≥ 0.7) detections are dominated by false alarms from Poisson noise. Modern critiques (Parks 1998+) argue that Frank's claimed "holes" sit precisely in this marginal contrast regime, where statistical artifacts dominate. The geometry is delicate; a single instrument with poor cross-validation cannot decisively confirm such low-contrast features.

검출 효율 곡선은 Frank et al.이 직면한 근본적 어려움을 보여준다 — 깊이 ≳50%(인자 ≤ 0.5)에서 참 검출률은 가파르게 상승하지만, 얕은 대비(≳70%, 인자 ≥ 0.7)에서는 Poisson 잡음에 의한 오탐이 우세하다. 현대 비판(Parks 1998+)은 Frank가 주장한 "구멍"이 정확히 이 한계 대비 영역, 즉 통계 인공물이 지배하는 영역에 있다고 본다. 단일 계측의 교차 검증 없이는 그렇게 낮은 대비의 특징을 결정적으로 확인할 수 없다.

## Summary / 요약

| Concept / 개념 | This Paper / 이 논문 (1988) | Modern Equivalent / 현대 등가물 |
|---|---|---|
| Spin-scan imaging cadence / 스핀 스캔 영상 주기 | 12 min / 30°×30°, 0.25° pixel; mechanical mirror step | Polar UVI (1996): CCD frame transfer, ~37 s cadence; IMAGE FUV (2000): 2-min cadence wide-field |
| Auroral oval projection / 오로라 오벌 투영 | Hand-drawn coastlines overlaid on photographic prints | Modern: AACGM coordinates + WMM for B-field; ~10⁴ image archive at NASA SPDF |
| Dayglow background statistics / 데이글로우 배경 통계 | Poisson 1-σ excess interpreted as small-comet water clouds | Refuted: shot-noise + dark-current + cosmic-ray hits (Parks et al. 1998); standard photon-counting noise budget |
| PSBL ↔ poleward arc mapping / PSBL → 극측 아크 매핑 | Tsyganenko-Usmanov (1982) field model | Modern T96, T01, T05 models; assimilative SuperMAG/AMPERE-driven mapping |
| Theta aurora / 세타 오로라 | Discovered as TPA during northward IMF | Routinely modelled in global MHD simulations (BATS-R-US, OpenGGCM); IMF By-driven topology |
| Geocorona Chamberlain fit / 지구코로나 적합 | $T = 1050$ K, $n_c = 44{,}000$ atoms/cm³, $r_c = 1.08 R_E$ | Still the canonical reference; refined by TIMED/SABER, IMAGE GEO, MAVEN/IUVS |

**Final note / 마무리**: The DE 1 spin-scan paradigm is a textbook case of trading time-resolution for spatial coverage with a mechanical scanning system. Modern UV imagers use staring CCDs with much higher quantum efficiency and effective area, but the basic geometric and photometric principles — angular pixel size vs. orbital altitude, Rayleigh photon flux, Chamberlain-style exospheric fits, and the imperative of stray-light suppression — remain unchanged.

DE 1 스핀 스캔 패러다임은 기계적 스캔 시스템으로 시간 해상도를 공간 커버리지와 교환하는 교과서적 사례다. 현대 UV 영상기는 응시(stare)형 CCD로 양자 효율과 유효 면적이 훨씬 크지만, 기본 기하·광도 원리(각픽셀 크기 vs. 궤도 고도, 레일리 광자 플럭스, Chamberlain 방식 외기권 적합, 잡광 억제의 필요성)는 변하지 않았다.